# 第 7 周 - 笔记本 2：测试基础模型

## 目标
Test the base Llama 3.2 model BEFORE fine-tuning to establish baseline:
1. 4位量化加载基础模型
2. 样品测试
3. 衡量基线绩效

预期结果：大约 110 美元的错误（太糟糕了！）

## 时间：5-10 分钟

**注意：** 此笔记本电脑需要 GPU。如果您本地没有 Google Colab，请在 Google Colab 上运行。

In [ ]:
import sys
sys.path.append('..')

import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from src.items import Item
from src.evaluator import evaluate
from src.config import config

# 负载环境
# Load environment
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 加载测试数据

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")
print(f"\nExample item:")
print(f"  Title: {test[0].title}")
print(f"  Price: ${test[0].price:.2f}")

## 负载基础模型（4 位量化）

In [ ]:
# 配置 4 位量化
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 设置填充标记
# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Model loaded in 4-bit")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 对样品产品进行测试

In [ ]:
def predict_base_llama(item: Item) -> str:
    """
    Predict price using base Llama model
    """
    # 创建提示
    # Create prompt
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)
    
    prompt = item.test_prompt()
    
    # 标记化
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 产生
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 解码
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取完成
    # Extract completion
    completion = response.split(config.PREFIX)[-1].strip()
    
    return completion

In [ ]:
# 测试几个例子
# Test on a few examples
print("Testing base model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    prediction = predict_base_llama(item)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)

## 对测试集的全面评估

In [ ]:
# 在完整的测试集上进行评估
# Evaluate on full test set
results = evaluate(
    predict_base_llama,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("BASE MODEL RESULTS")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")

## 概括

✅ 基础模型测试完成！

**预期结果：**
- Base Llama 3.2 4 位：~$110 错误（太可怕了！）
- 模特不知道如何给产品定价
- 预测本质上是随机的

**为什么这么糟糕？**
- 基础模型未经过价格预测训练
- 没有有关产品定价的领域知识
- 需要对我们的具体任务进行微调

**下一步：** `03_finetune_colab.ipynb` - 使用 QLoRA 进行微调以实现约 40 美元的误差！